In [ ]:
import pandas as pd
import numpy as np
import plotnine as p9
import glob
import yaml
from plotnine_prism import *
import sys
sys.path.append('../')
from src.utils import load_data
from src.utils import bootstrapping
from src.utils import compute_pearson_top_n
from src.utils import compute_area_under_pearson_top_n

In [ ]:
with open("../config.yaml", "r") as stream:
    DATASET_INFO = yaml.safe_load(stream)
DATASET_INFO

In [ ]:
scores = []
for dataset, dataset_name in DATASET_INFO["DATASET_NAME"].items():
    if "TCGA" in dataset or 'Xenium' in dataset: continue
    with open(f"../{dataset}/config_dataset.yaml", "r") as stream:
        config_dataset = yaml.safe_load(stream)
    models = config_dataset["MODEL"]
    
    for model in ["DeepSpot"]:

        top_model = pd.read_csv(f"../{dataset}/out_benchmark/evaluation/{model}/top_model_per_test_sample.csv")
        model_dict = top_model[["test_sample", "model"]].set_index("test_sample").to_dict()['model']
    
        
        for test in model_dict.keys():
        
            file = f"../{dataset}/out_benchmark/evaluation/{test}/gene_scores/test/{model}/{model_dict[test]}.csv"
            score = pd.read_csv(file)
            score = score.set_index("index")
            score.columns = ["pearson"]
            score["test_sample"] = test
            score["model"] = model
            score['dataset'] = dataset
            scores.append(score)


scores = pd.concat(scores)
scores["gene"] = scores.index
scores = scores.reset_index(drop=True)
scores = scores.groupby(["dataset", "gene"]).pearson.agg("mean").reset_index()
scores.pearson = abs(scores.pearson)
scores["z_score"] = scores.groupby("dataset").pearson.transform(lambda x: (x - x.mean()) / x.std())
scores

In [ ]:
scores.z_score.plot.hist()

In [ ]:
from gseapy import barplot, dotplot
import decoupler as dc
import gseapy as gp

In [ ]:
def get_pathways(gene_list):
    
    gene_sets = [
        #"GO_Molecular_Function_2025",
        #"GO_Cellular_Component_2025",
        "GO_Biological_Process_2025",
        #"Cancer_Cell_Line_Encyclopedia",
        #"KEGG_2021_Human",
    ]
    gene_sets = [f"../ressources/enrichr/{f}.gmt" for f in gene_sets]
    enr = gp.enrichr(gene_list=gene_list,
                     gene_sets=gene_sets,
                     organism='human',
                     outdir=None,  # don't write to disk
                     )
    out = enr.results[enr.results["Adjusted P-value"] < 0.05]
    return out

In [ ]:
good_out = scores.groupby("dataset").apply(lambda x: get_pathways(x.gene[x.z_score > np.percentile(x.z_score , 90)])).reset_index()
good_terms = good_out.groupby("Term").dataset.count().sort_values()
good_terms = good_terms[good_terms >= 2].index.values ##
good_out = good_out[good_out.Term.isin(good_terms)]
good_out["term_type"] = "good"

In [ ]:
good_out["go_term"] = good_out["Term"].apply(lambda x: x.split(" (")[1].replace(")", ""))
good_out

In [ ]:
good_out[["go_term"]].drop_duplicates().to_csv("resources/good_terms.csv", index=False)

In [ ]:
bad_out = scores.groupby("dataset").apply(lambda x: get_pathways(x.gene[x.z_score < 0])).reset_index()
bad_terms = bad_out.groupby("Term").dataset.count().sort_values()
bad_terms = bad_terms[bad_terms >= 3].index.values
bad_out = bad_out[bad_out.Term.isin(bad_terms)]
bad_out["term_type"] = "bad"
#bad_out